In [ ]:
#Qué hace: genera una Tabla 1 descriptiva estratificada por RCS (sí/no).
#Clave: caracterización basal; no inferencia causal.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# -----------------------------
# Paths
# -----------------------------
COHORT_PATH  = "04_cohorte_base_T0.parquet"
OUTCOME_PATH = "10_respuesta_clinica_sostenida.parquet"
OUT_TABLE1   = "11_table1.csv"

KEYS = ["subject_id", "hadm_id", "icu_stay_id"]

In [3]:
# -----------------------------
# 1) Load data
# -----------------------------
df_raw = pd.read_parquet(COHORT_PATH)
df_out = pd.read_parquet(OUTCOME_PATH)

In [4]:
# -----------------------------
# 2) Collapse cohort to 1 row per stay
# -----------------------------
df_raw["t0_antibiotic"] = pd.to_datetime(df_raw["t0_antibiotic"], errors="coerce")

df = (
    df_raw
    .sort_values(KEYS + ["t0_antibiotic"])
    .drop_duplicates(KEYS, keep="first")
)

assert df.duplicated(KEYS).sum() == 0

In [5]:
# -----------------------------
# 3) Outcome: ensure 1 row per stay
# -----------------------------
df_out = (
    df_out
    .groupby(KEYS, as_index=False)
    .agg(ever_rcs=("ever_rcs", "max"))
)

In [6]:
# -----------------------------
# 4) Merge baseline + outcome
# -----------------------------
df = df.merge(df_out, on=KEYS, how="left")
df["ever_rcs"] = df["ever_rcs"].fillna(0).astype(int)

print("Final N stays:", len(df))
print("Outcome Yes:", df["ever_rcs"].sum())
print("Outcome No:", (df["ever_rcs"] == 0).sum())

Final N stays: 21080
Outcome Yes: 7163
Outcome No: 13917


In [7]:
# -----------------------------
# 5) Build Table 1
# -----------------------------
rows = []

def add_row(var, total, yes, no):
    rows.append({
        "Variable": var,
        "Total": total,
        "Outcome=Yes": yes,
        "Outcome=No": no
    })

# N
add_row(
    "N",
    len(df),
    (df["ever_rcs"] == 1).sum(),
    (df["ever_rcs"] == 0).sum()
)

# Age
def median_iqr(x):
    return f"{x.median():.1f} [{x.quantile(0.25):.1f}–{x.quantile(0.75):.1f}]"

add_row(
    "age (median [Q1–Q3])",
    median_iqr(df["age"]),
    median_iqr(df.loc[df["ever_rcs"]==1, "age"]),
    median_iqr(df.loc[df["ever_rcs"]==0, "age"])
)

# Gender
for g in df["gender"].dropna().unique():
    add_row(
        f"gender = {g}",
        (df["gender"]==g).sum(),
        f"{(df.query('ever_rcs==1')['gender']==g).sum()} "
        f"({(df.query('ever_rcs==1')['gender']==g).mean()*100:.1f}%)",
        f"{(df.query('ever_rcs==0')['gender']==g).sum()} "
        f"({(df.query('ever_rcs==0')['gender']==g).mean()*100:.1f}%)"
    )

# Site
for s in df["site"].value_counts().index:
    add_row(
        f"site = {s}",
        (df["site"]==s).sum(),
        f"{(df.query('ever_rcs==1')['site']==s).sum()} "
        f"({(df.query('ever_rcs==1')['site']==s).mean()*100:.1f}%)",
        f"{(df.query('ever_rcs==0')['site']==s).sum()} "
        f"({(df.query('ever_rcs==0')['site']==s).mean()*100:.1f}%)"
    )

# Organism
for o in df["organism"].value_counts().index:
    add_row(
        f"organism = {o}",
        (df["organism"]==o).sum(),
        f"{(df.query('ever_rcs==1')['organism']==o).sum()} "
        f"({(df.query('ever_rcs==1')['organism']==o).mean()*100:.1f}%)",
        f"{(df.query('ever_rcs==0')['organism']==o).sum()} "
        f"({(df.query('ever_rcs==0')['organism']==o).mean()*100:.1f}%)"
    )

In [8]:
# -----------------------------
# 6) Save
# -----------------------------
table1 = pd.DataFrame(rows)
table1.to_csv(OUT_TABLE1, index=False)
print("Saved Table 1:", OUT_TABLE1)

Saved Table 1: 11_table1.csv


In [9]:
table1

,Variable,Total,Outcome=Yes,Outcome=No
0,N,21080,7163,13917
1,age (median [Q1–Q3]),65.0 [54.0–75.0],64.0 [53.0–74.0],65.0 [54.0–76.0]
2,gender = F,9130,2934 (41.0%),6196 (44.5%)
3,gender = M,11950,4229 (59.0%),7721 (55.5%)
4,site = urine,4526,1356 (18.9%),3170 (22.8%)
...,...,...,...,...
401,organism = salmonella typhi,1,0 (0.0%),1 (0.0%)
402,organism = positive for group b beta streptococci,1,0 (0.0%),1 (0.0%)
403,organism = eggerthella lenta,1,1 (0.0%),0 (0.0%)
404,organism = cryptosporidium parvum oocysts seen,1,1 (0.0%),0 (0.0%)
